# 🚀 Chapter 07-01b: Build a Chatbot with Intent Routing

---

## 📖 The Mission

Build a customer service chatbot that **routes** questions to different specialists:
- 📦 Order questions → Order handler
- 🔧 Technical questions → Tech support
- 💬 General questions → General assistant

This is your first LangGraph application!

---

## 🧪 Step 1: Setup

In [ ]:
# !pip install langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

load_dotenv()
print("✅ Imports ready!")

## 🧪 Step 2: Define the State

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # Append reducer
    intent: str  # Classified intent

print("✅ State defined with messages (append) and intent")

## 🧪 Step 3: Define Nodes

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Node 1: Classify user intent
def classify_intent(state: ChatState) -> dict:
    last_msg = state["messages"][-1].content
    classification = llm.invoke([
        SystemMessage(content="""Classify the user message into one of these categories.
        Reply with ONLY the category name:
        - order (about orders, shipping, tracking, returns)
        - technical (about product issues, bugs, how-to)
        - general (greetings, other questions)"""),
        HumanMessage(content=last_msg)
    ])
    intent = classification.content.strip().lower()
    print(f"  🏷️ Classified intent: {intent}")
    return {"intent": intent}

# Node 2: Handle order questions
def handle_order(state: ChatState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are an order support specialist. Help with order tracking, shipping, and returns. Be concise and helpful."),
        *state["messages"]
    ])
    print(f"  📦 Order handler responded")
    return {"messages": [response]}

# Node 3: Handle technical questions
def handle_technical(state: ChatState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a technical support expert. Help diagnose and fix technical issues. Provide step-by-step solutions."),
        *state["messages"]
    ])
    print(f"  🔧 Tech handler responded")
    return {"messages": [response]}

# Node 4: Handle general questions
def handle_general(state: ChatState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a friendly customer service assistant. Help with general questions."),
        *state["messages"]
    ])
    print(f"  💬 General handler responded")
    return {"messages": [response]}

print("✅ All 4 nodes defined!")

## 🧪 Step 4: Define Router (Conditional Edge)

In [ ]:
def route_by_intent(state: ChatState) -> Literal["handle_order", "handle_technical", "handle_general"]:
    """Route to the appropriate handler based on classified intent."""
    intent = state.get("intent", "general")
    if "order" in intent:
        return "handle_order"
    elif "technical" in intent or "tech" in intent:
        return "handle_technical"
    else:
        return "handle_general"

print("✅ Router function defined!")

## 🧪 Step 5: Build the Graph

In [ ]:
# Build the graph
graph = StateGraph(ChatState)

# Add nodes
graph.add_node("classify", classify_intent)
graph.add_node("handle_order", handle_order)
graph.add_node("handle_technical", handle_technical)
graph.add_node("handle_general", handle_general)

# Set entry point
graph.set_entry_point("classify")

# Conditional edges from classifier
graph.add_conditional_edges("classify", route_by_intent)

# All handlers go to END
graph.add_edge("handle_order", END)
graph.add_edge("handle_technical", END)
graph.add_edge("handle_general", END)

# Compile!
app = graph.compile()
print("✅ Graph compiled!")

## 🧪 Step 6: Visualize the Graph

In [ ]:
# Print the graph structure (ASCII)
try:
    print(app.get_graph().draw_ascii())
except Exception:
    print("Graph: START → classify → [order|technical|general] → END")

## 🧪 Step 7: Test the Chatbot!

In [ ]:
def chat(message: str):
    print(f"\n{'='*50}")
    print(f"🧑 User: {message}")
    print(f"{'='*50}")
    result = app.invoke({"messages": [HumanMessage(content=message)]})
    final = result["messages"][-1].content
    print(f"\n🤖 Response: {final[:300]}")
    return final

# Test routing to different handlers
chat("Where is my order #12345?")

In [ ]:
chat("My app keeps crashing when I click the export button")

In [ ]:
chat("Hi! What products do you sell?")

## ✍️ Practice Challenge

Add a **"billing"** intent and handler to the graph. The billing handler should deal with payment issues, invoices, and refunds.

<details>
<summary>💡 Hint</summary>

1. Add `handle_billing` node function
2. Add it to the graph: `graph.add_node("handle_billing", handle_billing)`
3. Update `route_by_intent` to check for "billing" in intent
4. Add edge: `graph.add_edge("handle_billing", END)`
5. Re-compile with `graph.compile()`

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 StateGraph(State) → add_node → add_edge → compile → invoke.   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 add_conditional_edges() enables routing based on state.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Nodes return partial state — only the keys they change.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Annotated[list, add_messages] appends (not replaces) messages. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 END signals the graph is finished — returns final state.       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Control Flow — Cycles, Loops & Human-in-the-Loop!

---

### 🎉 Your First LangGraph App!
A chatbot that thinks before routing — that's intelligent design. 🚀